#### This dataset was collected from FBI website. The website provides NIBRS data based on crime incidents that will allows to see all crimes occurced across 51 states reporting crime statistics on each incident as well as seperate offense level stats on same incident. There are 28 tables of data that have chained relatioship among variables defining so many characterstics of each crime incident and offense types based on Bias,Weapon used,Drug involvement, location of incident, property and much more. Since it can be difficult to merge tables with various indexes for a key variable, I made sure that the right table had a unique key and used a left join to combine the tables. This dataset was created after extensive joins of the tables using a single key variable. New Jersey(2022) data was used to prepare the dataset for our ml modelling problem. 

Link to the resource for dataset : https://cde.ucr.cjis.gov/LATEST/webapp/#

In [253]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

##### Using Age table

In [254]:
df_age = pd.read_csv('dataset/NIBRS_AGE.csv')
df_age.head(2)

,age_id,age_code,age_name
0,1,NN,Under 24 Hours
1,2,NB,1-6 Days Old


In [256]:
# Check the age values
df_age['age_name'].value_counts()

age_name
Under 24 Hours       1
1-6 Days Old         1
72 Years Old         1
71 Years Old         1
70 Years Old         1
                    ..
27 Years Old         1
26 Years Old         1
25 Years Old         1
24 Years Old         1
Over 98 Years Old    1
Name: count, Length: 104, dtype: int64

##### Using the ethnicity table

In [257]:
df_etnicity = pd.read_csv('dataset/NIBRS_ETHNICITY.csv')
df_etnicity.head(2)

,ethnicity_id,ethnicity_code,ethnicity_name
0,10,H,Hispanic or Latino
1,20,N,Not Hispanic or Latino


##### Using the race table

In [258]:
df_race = pd.read_csv('dataset/REF_RACE.csv')
df_race.head(2)

,race_id,race_code,race_desc,sort_order,start_year,end_year,notes
0,98,U,Unknown,98,NaN,NaN,Race is explicitly unknown
1,10,W,White,10,NaN,NaN,NaN


##### Using Victim type table

In [259]:
df_victim_type = pd.read_csv('dataset/NIBRS_VICTIM_TYPE.csv')
df_victim_type.head(2)

,victim_type_id,victim_type_code,victim_type_name
0,1,B,Business
1,2,F,Financial Institution


##### Using the Victim table

In [260]:
df_victim = pd.read_csv('dataset/NIBRS_VICTIM.csv')
df_victim.head(2)

,data_year,victim_id,incident_id,victim_seq_num,victim_type_id,assignment_type_id,activity_type_id,outside_agency_id,age_id,age_num,sex_code,race_id,ethnicity_id,resident_status_code,age_range_low_num,age_code_range_high
0,2022,177976762,160891554,1,4,NaN,NaN,NaN,43,40,M,10,20,R,40.0,NaN
1,2022,177976763,160891555,1,1,NaN,NaN,NaN,104,NS,X,99,50,NaN,NaN,NaN


In [261]:
print(df_victim.shape)

# We have populated victim age,ethnicity,race from the above tables
df_victim = df_victim.merge(df_age[['age_id','age_name']],on='age_id',how='left')
df_victim = df_victim.merge(df_etnicity[['ethnicity_id','ethnicity_name']],on='ethnicity_id',how='left')
df_victim = df_victim.merge(df_race[['race_id','race_desc']],on='race_id',how='left')
df_victim = df_victim.merge(df_victim_type[['victim_type_id','victim_type_name']],on='victim_type_id',how='left')
df_victim.rename(columns={'age_name':'victim_age',\
    'sex_code':'victim_sex','ethnicity_name':'victim_ethnicity'\
        ,'race_desc':'victim_race'},inplace=True)


print(df_victim.shape)

(175595, 16)
(175595, 20)


We are filtering unique incidents because first we are going to merge on incident_id(incident_levle)

In [262]:
incident_id_counts = df_victim['incident_id'].value_counts()

# Filter rows where incident_id occurs exactly once
unique_incident_ids = incident_id_counts[incident_id_counts == 1].index
df_victim = df_victim[df_victim['incident_id'].isin(unique_incident_ids)]

df_victim.shape

(156669, 20)

In [263]:
# Taking only important feature from victim table
df_victim = df_victim[['victim_id','incident_id','victim_type_name','victim_age','victim_race','victim_sex','victim_ethnicity']]

##### Using injury table

In [265]:
df_injury = pd.read_csv('dataset/NIBRS_INJURY.csv')
df_injury.head(2)

,injury_id,injury_code,injury_name
0,1,B,Apparent Broken Bones
1,2,I,Possible Internal Injury


##### Using victim injury table

In [266]:
df_victim_injury = pd.read_csv('dataset/NIBRS_VICTIM_INJURY.csv')
df_victim_injury.head(2)

,data_year,victim_id,injury_id
0,2022,177976767,7
1,2022,177976772,5


In [157]:
# df_victim_injury= df_victim_injury.merge(df_injury[['injury_id','injury_name']],on='injury_id',how='left')

# df_victim = df_victim.merge(df_victim_injury[['victim_id','injury_name']],on='victim_id',how='left')

##### using relationship table

In [267]:
df_relationship = pd.read_csv('dataset/NIBRS_RELATIONSHIP.csv')
df_relationship.head(2)

,relationship_id,relationship_code,relationship_name
0,1,AQ,Victim Was Acquaintance
1,2,BE,Victim Was Babysittee


##### Using victim offender rel table

In [269]:
df_victim_offender = pd.read_csv('dataset/NIBRS_VICTIM_OFFENDER_REL.csv')
df_victim_offender.head(2)

,data_year,victim_id,offender_id,relationship_id,nibrs_victim_offender_id
0,2022,177976767,182684563,1,NaN
1,2022,179957439,184638493,26,NaN


This will be our final table, so we will first make subtables and merge all those tables into this table ex: victim_table,offender_table,offense_table. So, to make it feasible for merging these tables we are only taking unique victim and offender combination

In [270]:
import pandas as pd


# count the occurrences of each victim_id and offender_id
victim_counts = df_victim_offender.groupby('victim_id').size().reset_index(name='victim_count')
offender_counts = df_victim_offender.groupby('offender_id').size().reset_index(name='offender_count')

# merge the counts back into the original dataFrame
merged_df = pd.merge(df_victim_offender, victim_counts, on='victim_id', how='left')
merged_df = pd.merge(merged_df, offender_counts, on='offender_id', how='left')

# filter rows where both victim_id and offender_id occur exactly once
df_victim_offender = merged_df[(merged_df['victim_count'] == 1) & (merged_df['offender_count'] == 1)]

# drop the count columns
df_victim_offender = df_victim_offender.drop(['victim_count', 'offender_count'], axis=1)

df_victim_offender.head(2)


,data_year,victim_id,offender_id,relationship_id,nibrs_victim_offender_id
0,2022,177976767,182684563,1,NaN
1,2022,179957439,184638493,26,NaN


In [271]:
# taking only important variables
df_victim_offender = df_victim_offender.merge(df_relationship[['relationship_id','relationship_name']],on='relationship_id',how='left')

We are filtering unique incidents because first we are going to merge on incident_id(incident_levle)

In [272]:
incident_id_counts = df_victim['incident_id'].value_counts()

# filter rows where incident_id occurs exactly once
unique_incident_ids = incident_id_counts[incident_id_counts == 1].index
df_victim = df_victim[df_victim['incident_id'].isin(unique_incident_ids)]

df_victim.shape

(156669, 7)

We are doing left join here based on victim_id as key variable, since we don't have duplicate victim_ids we can easily join them and it will not effect the shape of our main table

In [273]:
df_victim_offender = df_victim_offender.merge(df_victim,on='victim_id',how='left')

In [274]:
df_victim_offender.shape

(47341, 12)

##### using the offender table

In [276]:
df_offender = pd.read_csv('dataset/NIBRS_OFFENDER.csv')
df_offender.head(2)

,data_year,offender_id,incident_id,offender_seq_num,age_id,age_num,sex_code,race_id,ethnicity_id,age_range_low_num,age_range_high_num
0,2022,182684558,160891554,0,104,NS,X,99,50,NaN,NaN
1,2022,182684559,160891555,1,62,59,M,20,20,59.0,NaN


In [277]:
print(df_offender.shape)

# populating the age,ethnicity,race values for offender
df_offender = df_offender.merge(df_age[['age_id','age_name']],on='age_id',how='left')
df_offender = df_offender.merge(df_etnicity[['ethnicity_id','ethnicity_name']],on='ethnicity_id',how='left')
df_offender = df_offender.merge(df_race[['race_id','race_desc']],on='race_id',how='left')
df_offender.rename(columns={'age_name':'offender_age',\
    'sex_code':'offender_sex','ethnicity_name':'offender_ethnicity'\
        ,'race_desc':'offender_race'},inplace=True)


print(df_offender.shape)

(174928, 11)
(174928, 14)


In [278]:
# taking only the important variables from offender table
df_victim_offender = df_victim_offender.merge(df_offender[['offender_id','offender_age','offender_sex','offender_ethnicity','offender_race']],on='offender_id',how='left')

In [279]:
df_victim_offender.columns

Index(['data_year', 'victim_id', 'offender_id', 'relationship_id',
       'nibrs_victim_offender_id', 'relationship_name', 'incident_id',
       'victim_type_name', 'victim_age', 'victim_race', 'victim_sex',
       'victim_ethnicity', 'offender_age', 'offender_sex',
       'offender_ethnicity', 'offender_race'],
      dtype='object')

##### Using offense table

In [281]:
df_offense_type = pd.read_csv('dataset/NIBRS_OFFENSE_TYPE.csv')
df_offense_type.head(2)

,offense_code,offense_name,crime_against,ct_flag,hc_flag,hc_code,offense_category_name,offense_group
0,09A,Murder and Nonnegligent Manslaughter,Person,f,t,01,Homicide Offenses,A
1,09B,Negligent Manslaughter,Person,f,t,,Homicide Offenses,A


##### Using location_type table

In [282]:
df_location = pd.read_csv('dataset/NIBRS_LOCATION_TYPE.csv')
df_location.head(2)

,location_id,location_code,location_name
0,1,37,Abandoned/Condemned Structure
1,2,1,Air/Bus/Train Terminal


##### Using offense table

In [283]:
df_offense = pd.read_csv('dataset/NIBRS_OFFENSE.csv')
df_offense.head(2)

,data_year,offense_id,incident_id,offense_code,attempt_complete_flag,location_id,num_premises_entered,method_entry_code
0,2022,192326192,160891554,290,C,25,NaN,NaN
1,2022,192326193,160891555,23C,C,17,NaN,NaN


In [284]:
# fill the offense_name,crime_against,...location values into offense table
df_offense = df_offense.merge(df_offense_type,on='offense_code',how='left')
df_offense = df_offense.merge(df_location[['location_id','location_name']],on='location_id',how='left')


In [285]:
#taking only the important variables
df_offense = df_offense[['data_year','offense_id','incident_id','attempt_complete_flag',\
    'location_name','offense_name','crime_against','ct_flag','hc_flag','offense_category_name','offense_group']]

In [286]:
df_offense.head(2)

,data_year,offense_id,incident_id,attempt_complete_flag,location_name,offense_name,crime_against,ct_flag,hc_flag,offense_category_name,offense_group
0,2022,192326192,160891554,C,Highway/Road/Alley/Street/Sidewalk,Destruction/Damage/Vandalism of Property,Property,f,t,Destruction/Damage/Vandalism of Property,A
1,2022,192326193,160891555,C,Department/Discount Store,Shoplifting,Property,f,f,Larceny/Theft Offenses,A


##### Using the bias list table

In [287]:
df_bias_list = pd.read_csv('dataset/NIBRS_BIAS_LIST.csv')
df_bias_list.head(2)

,bias_id,bias_code,bias_category,bias_desc
0,11,11,Race/Ethnicity/Ancestry,Anti-White
1,12,12,Race/Ethnicity/Ancestry,Anti-Black or African American


In [176]:
df_bias_list['bias_id'].value_counts().max()

1

##### Using the bias motivation table

In [288]:
df_bias_motivation = pd.read_csv('dataset/NIBRS_BIAS_MOTIVATION.csv')
df_bias_motivation.head(2)

,data_year,bias_id,offense_id
0,2022,88,192326192
1,2022,88,192326193


In [289]:
# fill the bias category,bias desc based on bias id
df_bias_motivation = df_bias_motivation.merge(df_bias_list[['bias_id','bias_category','bias_desc']],on='bias_id',how='left')

In [290]:
df_bias_motivation.head(2)

,data_year,bias_id,offense_id,bias_category,bias_desc
0,2022,88,192326192,None/Unknown,None (no bias)
1,2022,88,192326193,None/Unknown,None (no bias)


In [291]:
# group by `offense_id` and aggregate the bias_category and bias_desc
grouped_df = df_bias_motivation.groupby('offense_id').agg({
    'bias_category': lambda x: ' | '.join(set(x)),
    'bias_desc': lambda x: ' | '.join(set(x))
}).reset_index()

# merge the aggregated dataFrame back to the original dataFrame on 'offense_id'
df_bias_motivation = pd.merge(df_bias_motivation, grouped_df, on='offense_id', how='left', suffixes=('', '_agg'))

# create new feature whether there are multiple biases 
df_bias_motivation['multiple_biases'] = df_bias_motivation.apply(lambda row: '|' in row['bias_category_agg'], axis=1)


In [292]:
# dropping duplicates so that it is feasible for merge
df_bias_motivation = df_bias_motivation.drop_duplicates(subset=['offense_id'])

In [293]:
df_bias_motivation.head(2)

,data_year,bias_id,offense_id,bias_category,bias_desc,bias_category_agg,bias_desc_agg,multiple_biases
0,2022,88,192326192,None/Unknown,None (no bias),None/Unknown,None (no bias),False
1,2022,88,192326193,None/Unknown,None (no bias),None/Unknown,None (no bias),False


In [294]:
# populating the bias descriptors from bias motivation table
df_offense = df_offense.merge(df_bias_motivation[['offense_id','bias_category_agg','bias_desc_agg','multiple_biases']],\
    on='offense_id',how='left')

In [295]:
df_offense.head(2)

,data_year,offense_id,incident_id,attempt_complete_flag,location_name,offense_name,crime_against,ct_flag,hc_flag,offense_category_name,offense_group,bias_category_agg,bias_desc_agg,multiple_biases
0,2022,192326192,160891554,C,Highway/Road/Alley/Street/Sidewalk,Destruction/Damage/Vandalism of Property,Property,f,t,Destruction/Damage/Vandalism of Property,A,None/Unknown,None (no bias),False
1,2022,192326193,160891555,C,Department/Discount Store,Shoplifting,Property,f,f,Larceny/Theft Offenses,A,None/Unknown,None (no bias),False


##### Using property level

In [296]:
df_prop_loss = pd.read_csv('dataset/NIBRS_PROP_LOSS_TYPE.csv')
df_prop_loss.head(2)

,prop_loss_id,prop_loss_name,prop_loss_desc
0,1,NaN,NaN
1,2,Burned,Burned (includes damage caused in fighting the...


##### Using the prop_desc_type table

In [297]:
df_prop_desc_type = pd.read_csv('dataset/NIBRS_PROP_DESC_TYPE.csv')
df_prop_desc_type.head(2)

,prop_desc_id,prop_desc_name,prop_desc_code
0,1,Aircraft,1
1,2,Alcohol,2


##### Using the property desc table

In [298]:
df_prop_desc = pd.read_csv('dataset/NIBRS_PROPERTY_DESC.csv')
df_prop_desc.head(2)

,data_year,property_id,prop_desc_id,property_value,date_recovered,nibrs_prop_desc_id
0,2022,158652796,3,1.0,NaN,NaN
1,2022,158652797,44,160.0,2022-08-10,NaN


In [299]:
# fill property desc based on property desc id
df_prop_desc = df_prop_desc.merge(df_prop_desc_type[['prop_desc_id','prop_desc_name']],\
    on='prop_desc_id',how='left')

df_prop_desc.head(2)


,data_year,property_id,prop_desc_id,property_value,date_recovered,nibrs_prop_desc_id,prop_desc_name
0,2022,158652796,3,1.0,NaN,NaN,Automobile
1,2022,158652797,44,160.0,2022-08-10,NaN,Chemicals


##### Using the property table

In [300]:
df_property = pd.read_csv('dataset/NIBRS_PROPERTY.csv')
df_property.head(2),df_property.shape

(   data_year  property_id  incident_id  prop_loss_id  stolen_count  \
 0       2022    158652796    160891554             4           NaN   
 1       2022    158652797    160891555             5           NaN   
 
    recovered_count  
 0              NaN  
 1              NaN  ,
 (139396, 6))

In [301]:
# get unique property id counts so that we can get other variables based on property_id and merge this into main table
property_counts = df_prop_desc['property_id'].value_counts()
unique_property_counts =property_counts[property_counts == 1].index
df_prop_desc = df_prop_desc[df_prop_desc['property_id'].isin(unique_property_counts)]

In [302]:
# fill property_value, prop_desc_name 
df_property = df_property.merge(df_prop_desc[['property_id','property_value','prop_desc_name']],\
    on='property_id',how='left')

In [303]:
df_property.head(2)

,data_year,property_id,incident_id,prop_loss_id,stolen_count,recovered_count,property_value,prop_desc_name
0,2022,158652796,160891554,4,NaN,NaN,1.0,Automobile
1,2022,158652797,160891555,5,NaN,NaN,160.0,Chemicals


In [304]:
df_prop_loss.head(2)

,prop_loss_id,prop_loss_name,prop_loss_desc
0,1,NaN,NaN
1,2,Burned,Burned (includes damage caused in fighting the...


In [305]:
# fill prop_loss,prop_loss_desc
df_property = df_property.merge(df_prop_loss[['prop_loss_id','prop_loss_name','prop_loss_desc']],on='prop_loss_id',how='left')

In [306]:
df_victim_offender.head(2)

,data_year,victim_id,offender_id,relationship_id,nibrs_victim_offender_id,relationship_name,incident_id,victim_type_name,victim_age,victim_race,victim_sex,victim_ethnicity,offender_age,offender_sex,offender_ethnicity,offender_race
0,2022,177976767,182684563,1,NaN,Victim Was Acquaintance,160891559.0,Individual,65 Years Old,White,M,Not Hispanic or Latino,31 Years Old,M,Hispanic or Latino,White
1,2022,179957439,184638493,26,NaN,Victim was Ex-Spouse,162620859.0,Individual,55 Years Old,Black or African American,M,Not Hispanic or Latino,61 Years Old,F,Not Hispanic or Latino,Black or African American


In [307]:
df_victim_offender.shape

(47341, 16)

In [308]:
# merge the property table on incident_id 
df_victim_offender = df_victim_offender.merge(df_property[['incident_id','property_id','property_value','prop_desc_name'\
    ,'prop_loss_name','prop_loss_desc']],on='incident_id',how='left')

##### Using suspect using table

In [309]:
df_using_list = pd.read_csv('dataset/NIBRS_USING_LIST.csv')
df_using_list.head(2)

,suspect_using_id,suspect_using_code,suspect_using_name
0,1,A,Alcohol
1,2,C,Computer Equipment (Handheld Devices)


##### Using the Suspect using table

In [310]:
df_suspect_using = pd.read_csv('dataset/NIBRS_SUSPECT_USING.csv')
df_suspect_using.head(2)

,data_year,suspect_using_id,offense_id
0,2022,4,192326192
1,2022,4,192326193


In [311]:
# fill suspect_using_name using suspect_using_id
df_suspect_using = df_suspect_using.merge(df_using_list[['suspect_using_id','suspect_using_name']],on='suspect_using_id',how='left')

In [312]:
# get unique offense_id in df_suspect_using table

offense_id_counts = df_suspect_using['offense_id'].value_counts()

# filter rows where incident_id occurs exactly once
unique_offense_ids = offense_id_counts[offense_id_counts == 1].index
df_suspect_using = df_suspect_using[df_suspect_using['offense_id'].isin(unique_offense_ids)]


In [313]:
df_suspect_using

,data_year,suspect_using_id,offense_id,suspect_using_name
0,2022,4,192326192,Not Applicable
1,2022,4,192326193,Not Applicable
2,2022,4,192326195,Not Applicable
3,2022,4,192326196,Not Applicable
4,2022,4,192326197,Not Applicable
...,...,...,...,...
177975,2022,4,201506403,Not Applicable
177976,2022,4,201506448,Not Applicable
177977,2022,4,201506525,Not Applicable
177978,2022,4,201506605,Not Applicable


In [314]:
# we will merge this df_suspect_using table into df_offense table using 'offense_id
df_offense = df_offense.merge(df_suspect_using[['offense_id','suspect_using_name']],on='offense_id',how='left')

##### using the suspected_drug_type table

In [315]:
df_suspect_drug_type = pd.read_csv('dataset/NIBRS_SUSPECTED_DRUG_TYPE.csv')
df_suspect_drug_type.head(2)

,suspected_drug_type_id,suspected_drug_code,suspected_drug_name
0,1,A,Crack Cocaine
1,2,B,Cocaine


##### Using the suspected_drug table

In [316]:
df_suspect_drug = pd.read_csv('dataset/NIBRS_SUSPECTED_DRUG.csv')
df_suspect_drug.head(2)

,data_year,suspected_drug_type_id,property_id,est_drug_qty,drug_measure_type_id,nibrs_suspected_drug_id
0,2022,4,158652802,1.0,11,14152287
1,2022,4,158652809,56.0,9,14152288


In [317]:
df_suspect_drug.shape

(6475, 6)

In [318]:
# fill suspected_drug_name using suspected_drug_type_id
df_suspect_drug = df_suspect_drug.merge(df_suspect_drug_type[['suspected_drug_type_id','suspected_drug_name']],on='suspected_drug_type_id',how='left')

In [319]:
df_suspect_drug.head(2)

,data_year,suspected_drug_type_id,property_id,est_drug_qty,drug_measure_type_id,nibrs_suspected_drug_id,suspected_drug_name
0,2022,4,158652802,1.0,11,14152287,Heroin
1,2022,4,158652809,56.0,9,14152288,Heroin


##### Using drug measure type table

In [320]:
df_suspect_drug_measure = pd.read_csv('dataset/NIBRS_DRUG_MEASURE_TYPE.csv')
df_suspect_drug_measure.head(2)

,drug_measure_type_id,drug_measure_code,drug_measure_name
0,1,GM,Gram
1,2,KG,Kilogram


In [321]:
# fill drug_measure_name using drug_measure_type_id
df_suspect_drug = df_suspect_drug.merge(df_suspect_drug_measure[['drug_measure_type_id','drug_measure_name']],\
    on='drug_measure_type_id',how='left')


In [322]:
#get unique propery_id counts

property_id_counts = df_suspect_drug['property_id'].value_counts()

# filter rows where incident_id occurs exactly once
unique_property_counts = property_id_counts[property_id_counts == 1].index
df_suspect_drug = df_suspect_drug[df_suspect_drug['property_id'].isin(unique_property_counts)]


In [323]:
df_suspect_drug.head(2)

,data_year,suspected_drug_type_id,property_id,est_drug_qty,drug_measure_type_id,nibrs_suspected_drug_id,suspected_drug_name,drug_measure_name
0,2022,4,158652802,1.0,11,14152287,Heroin,Not Reported
3,2022,4,160129976,202.0,1,14350606,Heroin,Gram


In [324]:
df_victim_offender['property_id'].value_counts().max()

1

In [325]:
len(set(df_victim_offender['property_id'].unique()).difference(set(df_suspect_drug['property_id'].unique())))

17774

In [326]:
# merge the df_suspect_drug table into df_victim_offender on property_id
df_victim_offender = df_victim_offender.merge(df_suspect_drug[['property_id','est_drug_qty','suspected_drug_name','drug_measure_name']]\
    ,on='property_id',how='left')

##### Using weapon type table

In [327]:
df_weapon_type = pd.read_csv('dataset/NIBRS_WEAPON_TYPE.csv')
df_weapon_type.head(2)

,weapon_id,weapon_code,weapon_name,shr_flag
0,1,11,Firearm,t
1,2,11A,Firearm (Automatic),f


##### Using weapon table

In [328]:
df_weapon = pd.read_csv('dataset/NIBRS_WEAPON.csv')
df_weapon.head(2),df_weapon.shape

(   data_year  weapon_id  offense_id  nibrs_weapon_id
 0       2022         41   192326197              NaN
 1       2022         41   192326204              NaN,
 (40912, 4))

In [329]:
# fill weapon_name based on weapon_id
df_weapon = df_weapon.merge(df_weapon_type[['weapon_id','weapon_name']],on='weapon_id',how='left')

In [330]:
df_weapon.head(2)

,data_year,weapon_id,offense_id,nibrs_weapon_id,weapon_name
0,2022,41,192326197,NaN,Personal Weapons
1,2022,41,192326204,NaN,Personal Weapons


In [331]:
# offense_id_counts = df_weapon['offense_id'].value_counts()

# # Filter rows where incident_id occurs exactly once
# unique_offense_ids = offense_id_counts[offense_id_counts == 1].index
# df_weapon = df_weapon[df_weapon['offense_id'].isin(unique_offense_ids)]


In [332]:
# group by `offense_id` and aggregate the non-NaN weapon names
grouped_df = df_weapon.groupby('offense_id').agg({
    'weapon_name': lambda x: ' | '.join(set(x[x.notna()]))
}).reset_index()

# merge the aggregated dataFrame back to the original DataFrame on 'offense_id'
df_weapon = pd.merge(df_weapon, grouped_df, on='offense_id', how='left', suffixes=('', '_agg'))

# create a new column indicating whether there are multiple weapons
df_weapon['multiple_weapons'] = df_weapon['weapon_name_agg'].apply(lambda x: '|' in x if pd.notna(x) else False)


In [333]:
df_weapon.head(2)

,data_year,weapon_id,offense_id,nibrs_weapon_id,weapon_name,weapon_name_agg,multiple_weapons
0,2022,41,192326197,NaN,Personal Weapons,Personal Weapons,False
1,2022,41,192326204,NaN,Personal Weapons,Personal Weapons,False


In [334]:
#drop duplicates on offense id and merge weapon table into df_offense table using offense_id
df_weapon.drop_duplicates(subset=['offense_id'],inplace=True)
df_offense = df_offense.merge(df_weapon[['offense_id','weapon_name','weapon_name_agg','multiple_weapons']],on='offense_id',how='left')

In [335]:
df_offense.columns

Index(['data_year', 'offense_id', 'incident_id', 'attempt_complete_flag',
       'location_name', 'offense_name', 'crime_against', 'ct_flag', 'hc_flag',
       'offense_category_name', 'offense_group', 'bias_category_agg',
       'bias_desc_agg', 'multiple_biases', 'suspect_using_name', 'weapon_name',
       'weapon_name_agg', 'multiple_weapons'],
      dtype='object')

##### Using victim_offense table

In [336]:
df_victim_offense = pd.read_csv('dataset/NIBRS_VICTIM_OFFENSE.csv')
df_victim_offense.head(2)

,data_year,victim_id,offense_id
0,2022,177976762,192326192
1,2022,177976763,192326193


In [337]:
df_victim_offense.shape

(185567, 3)

In [338]:
# get unique victim_id counts and get the offense_id they are associated with

victim_id_counts = df_victim_offense['victim_id'].value_counts()

# ilter rows where incident_id occurs exactly once
unique_victim_counts = victim_id_counts[victim_id_counts == 1].index
df_victim_offense = df_victim_offense[df_victim_offense['victim_id'].isin(unique_victim_counts)]


In [339]:
df_victim_offense.shape

(166406, 3)

In [340]:
df_victim_offender.shape

(49156, 24)

In [341]:
# fill the offense_id for each victim_id so that we can finally merge df_offense table on offense-id
df_victim_offender = df_victim_offender.merge(df_victim_offense[['victim_id','offense_id']],on='victim_id',how='left')

In [342]:
df_offense.columns

Index(['data_year', 'offense_id', 'incident_id', 'attempt_complete_flag',
       'location_name', 'offense_name', 'crime_against', 'ct_flag', 'hc_flag',
       'offense_category_name', 'offense_group', 'bias_category_agg',
       'bias_desc_agg', 'multiple_biases', 'suspect_using_name', 'weapon_name',
       'weapon_name_agg', 'multiple_weapons'],
      dtype='object')

In [343]:
# taking all the neccessary variables from df_offense table
df_victim_offender = df_victim_offender.merge(df_offense[['offense_id','attempt_complete_flag','location_name','offense_name'\
    ,'crime_against','ct_flag','hc_flag','offense_category_name','offense_group','bias_category_agg',\
        'bias_desc_agg','multiple_biases','suspect_using_name','weapon_name','weapon_name','multiple_weapons']],on='offense_id',how='left')

In [344]:
df_victim_offender.head(2),df_victim_offender.shape

(   data_year  victim_id  offender_id  relationship_id  \
 0       2022  177976767    182684563                1   
 1       2022  179957439    184638493               26   
 
    nibrs_victim_offender_id        relationship_name  incident_id  \
 0                       NaN  Victim Was Acquaintance  160891559.0   
 1                       NaN     Victim was Ex-Spouse  162620859.0   
 
   victim_type_name    victim_age                victim_race  ... hc_flag  \
 0       Individual  65 Years Old                      White  ...       t   
 1       Individual  55 Years Old  Black or African American  ...       f   
 
     offense_category_name offense_group bias_category_agg   bias_desc_agg  \
 0        Assault Offenses             A      None/Unknown  None (no bias)   
 1  Larceny/Theft Offenses             A      None/Unknown  None (no bias)   
 
   multiple_biases  suspect_using_name       weapon_name       weapon_name  \
 0           False      Not Applicable  Personal Weapons  Persona

##### Using the month table

In [345]:
df_month = pd.read_csv('dataset/NIBRS_month.csv')
df_month.head(2)

,data_year,nibrs_month_id,agency_id,month_num,inc_data_year,reported_status,report_date,update_flag,orig_format,data_home,ddocname,did,month_pub_status
0,2022,44216376,12084,8,2022,R,NaN,NaN,X,NaN,2022_08_NJ0010100_2022020004_INC,160382786,NaN
1,2022,44216376,12084,8,2022,R,NaN,NaN,X,NaN,2022_08_NJ0010100_2022020103_INC,160382789,NaN


In [346]:
# get month num to incident level
df_month.drop_duplicates(subset=['nibrs_month_id'],inplace=True)

##### Using the agencies table

In [347]:
df_agency = pd.read_csv('dataset/agencies.csv')
df_agency.head(2)

,yearly_agency_id,agency_id,data_year,ori,legacy_ori,covered_by_legacy_ori,direct_contributor_flag,dormant_flag,dormant_year,reporting_type,...,nibrs_leoka_start_date,nibrs_ct_start_date,nibrs_multi_bias_start_date,nibrs_off_eth_start_date,covered_flag,county_name,msa_name,publishable_flag,participated,nibrs_participated
0,120832022,12083,2022,NJ0010000,NJ0010000,NaN,N,N,NaN,I,...,2021-01-01,2021-05-01,2021-05-01,2021-05-01,N,ATLANTIC,"Atlantic City-Hammonton, NJ",Y,Y,Y
1,120842022,12084,2022,NJ0010100,NJ0010100,NaN,N,N,NaN,I,...,2021-07-01,2021-07-01,2021-07-01,2021-07-01,N,ATLANTIC,"Atlantic City-Hammonton, NJ",Y,Y,Y


##### Using the arrest type table

In [348]:
df_arestee_type = pd.read_csv('dataset/NIBRS_ARREST_TYPE.csv')
df_arestee_type.head(2)

,arrest_type_id,arrest_type_code,arrest_type_name
0,1,O,On View
1,2,S,Summoned / Cited


##### Using the arrestee weapon table

In [349]:
df_arestee_weapon = pd.read_csv('dataset/NIBRS_ARRESTEE_WEAPON.csv')
df_arestee_weapon.head(2)

,data_year,arrestee_id,nibrs_arrestee_weapon_id,weapon_id
0,2022,48875320,NaN,51
1,2022,48875322,NaN,51


In [350]:
# get weapon_name using weapon id

df_arestee_weapon = df_arestee_weapon.merge(df_weapon_type[['weapon_id','weapon_name']],on='weapon_id',how='left')

##### Using arrestee table

In [351]:
df_arestee = pd.read_csv('dataset/NIBRS_ARRESTEE.csv')
df_arestee.head(2)

,data_year,arrestee_id,incident_id,arrestee_seq_num,arrest_date,arrest_type_id,multiple_indicator,offense_code,age_id,age_num,sex_code,race_id,ethnicity_id,resident_code,under_18_disposition_code,clearance_ind,age_range_low_num,age_range_high_num
0,2022,48875320,160891555,1,2022-08-10,1,N,23C,62,59,M,20,20,N,NaN,NaN,59.0,NaN
1,2022,48875322,160891557,1,2022-08-11,1,N,23C,53,50,M,10,40,N,NaN,NaN,50.0,NaN


In [352]:
#drop duplicates and get weapon_name each arrestee associated with
df_arestee_weapon.drop_duplicates(subset=['arrestee_id'],inplace=True)
df_arestee = df_arestee.merge(df_arestee_weapon[['arrestee_id','weapon_name']],on='arrestee_id',how='left')

In [353]:
# fill the arrest type for each arestee
df_arestee = df_arestee.merge(df_arestee_type[['arrest_type_id','arrest_type_name']],on='arrest_type_id',how='left')

In [354]:
# merge the arrest level stats to incident dataframe

#get number of arrest at incident level
num_arestee = df_arestee.groupby('incident_id')['arrestee_id'].size().reset_index(name='num_arestee')

#create a column with num_arestee
df_arestee = df_arestee.merge(num_arestee, on='incident_id', how='left')

df_arestee = df_arestee.rename(columns={'weapon_name': 'arrestee_weapon_name'})

df_arestee.head(2)

,data_year,arrestee_id,incident_id,arrestee_seq_num,arrest_date,arrest_type_id,multiple_indicator,offense_code,age_id,age_num,...,race_id,ethnicity_id,resident_code,under_18_disposition_code,clearance_ind,age_range_low_num,age_range_high_num,arrestee_weapon_name,arrest_type_name,num_arestee
0,2022,48875320,160891555,1,2022-08-10,1,N,23C,62,59,...,20,20,N,NaN,NaN,59.0,NaN,Unarmed,On View,1
1,2022,48875322,160891557,1,2022-08-11,1,N,23C,53,50,...,10,40,N,NaN,NaN,50.0,NaN,Unarmed,On View,1


##### Using the incident table

In [355]:
df_incident = pd.read_csv('dataset/NIBRS_incident.csv')
df_incident.head(2)

,data_year,agency_id,incident_id,nibrs_month_id,cargo_theft_flag,submission_date,incident_date,report_date_flag,incident_hour,cleared_except_id,cleared_except_date,incident_status,data_home,orig_format,did
0,2022,12084,160891554,44216376,f,2022-11-01 22:13:27.186,2022-08-09,f,10.0,6,NaN,ACCEPTED,NaN,X,160382786
1,2022,12084,160891555,44216376,f,2022-11-01 22:13:27.586,2022-08-10,f,14.0,6,NaN,ACCEPTED,NaN,X,160382789


In [356]:
# fill month,arrestee,agency info for each incident
df_incident = df_incident.merge(df_month[['nibrs_month_id','month_num']],on='nibrs_month_id')
df_incident = df_incident.merge(df_arestee[['incident_id','num_arestee','arrest_type_name','arrestee_weapon_name']],\
                                on='incident_id',how='left')
df_incident = df_incident.merge(df_agency[['agency_id','state_name','region_name','county_name']],on='agency_id',how='left')

In [357]:
df_incident.head(2)

,data_year,agency_id,incident_id,nibrs_month_id,cargo_theft_flag,submission_date,incident_date,report_date_flag,incident_hour,cleared_except_id,...,data_home,orig_format,did,month_num,num_arestee,arrest_type_name,arrestee_weapon_name,state_name,region_name,county_name
0,2022,12084,160891554,44216376,f,2022-11-01 22:13:27.186,2022-08-09,f,10.0,6,...,NaN,X,160382786,8,NaN,NaN,NaN,New Jersey,Northeast,ATLANTIC
1,2022,12084,160891555,44216376,f,2022-11-01 22:13:27.586,2022-08-10,f,14.0,6,...,NaN,X,160382789,8,1.0,On View,Unarmed,New Jersey,Northeast,ATLANTIC


In [358]:
df_victim_offender.columns

Index(['data_year', 'victim_id', 'offender_id', 'relationship_id',
       'nibrs_victim_offender_id', 'relationship_name', 'incident_id',
       'victim_type_name', 'victim_age', 'victim_race', 'victim_sex',
       'victim_ethnicity', 'offender_age', 'offender_sex',
       'offender_ethnicity', 'offender_race', 'property_id', 'property_value',
       'prop_desc_name', 'prop_loss_name', 'prop_loss_desc', 'est_drug_qty',
       'suspected_drug_name', 'drug_measure_name', 'offense_id',
       'attempt_complete_flag', 'location_name', 'offense_name',
       'crime_against', 'ct_flag', 'hc_flag', 'offense_category_name',
       'offense_group', 'bias_category_agg', 'bias_desc_agg',
       'multiple_biases', 'suspect_using_name', 'weapon_name', 'weapon_name',
       'multiple_weapons'],
      dtype='object')

In [359]:
# drop the null incident_ids these are formed when we done merging
df_victim_offender.dropna(subset=['incident_id'],inplace=True)

In [360]:
# fill all the incident info for each victim offender pair
df_victim_offender = df_victim_offender.merge(df_incident[['incident_id','incident_date','submission_date','incident_hour','month_num',\
                                      'num_arestee','arrest_type_name','arrestee_weapon_name'\
                                        ,'state_name','region_name','county_name']]\
                                            ,on='incident_id',how='left')

#### This Lat_Long table is not present in the NIBRS data. I have created this table using the county name from the internet

##### Using the lat long table

In [361]:
df_lat_long = pd.read_csv('dataset/LAT_LONG.csv')
df_lat_long.head(2)

,county_name,longitude,latitude
0,ATLANTIC,39.5333,-74.6869
1,BURLINGTON,39.8558,-74.6869


In [362]:
# merge the latitude and longitude
df_victim_offender = df_victim_offender.merge(df_lat_long[['county_name','latitude','longitude']],on='county_name',how='left')

In [363]:

# finally save the data file
df_victim_offender.to_csv('data.csv',index=None)